<a href="https://colab.research.google.com/github/anaisaoviedo-upb/Energia-2026/blob/main/Clasificacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicción de tipo clasificación

1. Preparación de Datos
2. División de los datos 70-30
3. Aprendizaje del Modelo
4. Evaluación del Modelo: exactitud
5. Guardar el modelo

* El despliegue se realiza en otro jupyter_notebook

In [ ]:
#Importamos librerías básicas
import pandas as pd # manipulacion dataframes
import numpy as np  # matrices y vectores
import matplotlib.pyplot as plt #gráfica

# 1. Preparación de Datos


In [ ]:
#Cargamos los datos
data = pd.read_excel("datos_limpios.xlsx", sheet_name=0)
data.head()

In [ ]:
#Conocemos los datos
data.info()

In [ ]:
#Corrección tipos de datos
data['Region']=data['Region'].astype('category')

# Completa

data.info()

In [ ]:
#Eliminamos variables irrelevantes para el modelo predictivo

data = data.drop(columns='Unnamed: 0') # Id de fila irrelevante
data = data.drop(columns='Energías Renovables_t+1') # Se desea predecir como categoría y no como número
data = data.drop(columns='Year') # Year no representa una característica energética del país sino una referencia temporal
data.head()

# **Transformaciones**

In [ ]:
#Dummies para las variables predictoras categóricas
data = pd.get_dummies(data, columns=['Region'], drop_first=False, dtype=int)

data.head()

# 2. División 70-30


In [ ]:
#División 70-30
from sklearn.model_selection import train_test_split
X = data.drop("Nivel transicion energetica_t+1", axis = 1) # Variables predictoras
Y = data['Nivel transicion energetica_t+1'] #Variable objetivo
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, stratify=Y) #Muestreo estratificado
Y_train.value_counts().plot(kind='bar')

# 3. Aprendizaje con el 70% y Evaluación con el 30%


# **Tree**
No se normaliza

In [ ]:
#Creación del modelo con el conjunto de entrenamiento
from sklearn.tree import DecisionTreeClassifier #DecisionTreeRegressor

modelTree = DecisionTreeClassifier(criterion='gini', min_samples_leaf=2, max_depth=20) #gini, entropy
modelTree.fit(X_train, Y_train) #70% train



In [ ]:
from sklearn.tree import plot_tree
plt.figure(figsize=(30,20)) #Tamaño de la imagen
plot_tree(modelTree, feature_names=X_train.columns.values, class_names=modelTree.classes_, rounded=True,  fontsize=12, filled=True)
plt.show()

In [ ]:
#Evaluación 30% (X_test)
from sklearn import metrics

Y_pred = modelTree.predict(X_test) #30% Test
print(Y_pred)


In [ ]:
#Exactitud: Correctos/Total
exactitud=metrics.accuracy_score(y_true=Y_test, y_pred=Y_pred)
print(exactitud)

# **Random Forest**

In [ ]:
#Random Forest
from sklearn.ensemble import RandomForestClassifier

model_rf= RandomForestClassifier(n_estimators=100,  max_samples=0.9, criterion='gini',
                              max_depth=20, min_samples_leaf=2)
model_rf.fit(X_train, Y_train) #70%

In [ ]:
#Importancia de las características
importances = model_rf.feature_importances_
feature_names = X_train.columns
forest_importances = pd.Series(importances, index=feature_names)

fig, ax = plt.subplots()
forest_importances.sort_values(ascending=False).plot.bar(ax=ax)
ax.set_title("Importancia de las características (Random Forest)")
ax.set_ylabel("Importancia Media según el criterion")
fig.tight_layout()
plt.show()

In [ ]:
#Evaluación de RandomForest con 30%
from sklearn import metrics

Y_pred = model_rf.predict(X_test) #30%

#Exactitud: Correctos/Total
exactitud=metrics.accuracy_score(y_true=Y_test, y_pred=Y_pred)
print(exactitud)




# **KNN**
* Normalización

In [ ]:
#Normalizacion las variables numéricas (las dummies no se normalizan)
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()
variables_numericas=['Petróleo_t','Gas Natural_t','Carbón_t','Energía Hidroeléctrica_t','Energías Renovables_t']

min_max_scaler.fit(data[variables_numericas]) #Ajuste de los parametros sobre 100% de los datos (data): max - min

#Se aplica la normalización a 70%  y 30%
X_train[variables_numericas]= min_max_scaler.transform(X_train[variables_numericas]) #70%
X_test[variables_numericas]= min_max_scaler.transform(X_test[variables_numericas]) #30%
X_train.head()

In [ ]:
#Aprendizaje KNN con 70%
from sklearn.neighbors  import KNeighborsClassifier #KNeighborsRegressor

modelKnn = KNeighborsClassifier(n_neighbors=1, metric='euclidean')#euclidean, minkowski
modelKnn.fit(X_train, Y_train) #70%

In [ ]:
#Evaluación de Knn con 30%
from sklearn import metrics

Y_pred = modelKnn.predict(X_test) #30%

#Exactitud: Correctos/Total
exactitud=metrics.accuracy_score(y_true=Y_test, y_pred=Y_pred)
print(exactitud)



# 4. Guardamos el mejor modelo


**Selecion del modelo:**
1. Calidad de los modelos
2. Complejidad computacional: Tree mas liviano computacionalmente

In [ ]:
#Selección del modelo final
modelo_final=modelKnn

Se entrena modelo final con 100% de los datos (X,Y)

In [ ]:
# Si selecciona Knn -> se debe normalzar X (100% de los datos)
# Si selecciona Tree o RF no se normaliza y se comenta la siguiente línea
X[variables_numericas]= min_max_scaler.transform(X[variables_numericas])

In [ ]:
#Entrenamos modelo final
modelo_final.fit(X, Y) #100%

In [ ]:
import pickle
filename = 'modelo-class.pkl'
variables= X.columns._values
pickle.dump([modelo_final,min_max_scaler,variables], open(filename, 'wb'))


